# Examine GP Activation NPZ

Inspect aggregated activation logs (e.g., `GP/datasets/raw/<run_id>/activations_all.npz`).

This notebook prints summary statistics and plots seaborn histograms per signal. If epoch metadata is present (`<signal>_epoch`), counts per epoch are shown.

In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Point to your aggregated NPZ
# Example: Path("./GP/datasets/raw/<run_id>/activations_all.npz")
npz_path = Path("./GP/datasets/raw/activations_all.npz")
assert npz_path.exists(), f"NPZ not found: {npz_path}"
data = np.load(npz_path)
signals = [k for k in data.files if not k.endswith("_epoch")]
print(f"Loaded signals: {signals}")

In [ ]:
def summarize_array(arr: np.ndarray, name: str):
    arr = arr.astype(np.float32)
    stats = {
        "count": arr.size,
        "min": float(np.min(arr)) if arr.size else None,
        "max": float(np.max(arr)) if arr.size else None,
        "mean": float(np.mean(arr)) if arr.size else None,
        "std": float(np.std(arr)) if arr.size else None,
        "p1": float(np.percentile(arr, 1)) if arr.size else None,
        "p50": float(np.percentile(arr, 50)) if arr.size else None,
        "p99": float(np.percentile(arr, 99)) if arr.size else None,
    }
    print(f"\n{name} summary:")
    for k, v in stats.items():
        print(f"  {k}: {v}")

def per_epoch_counts(sig: str):
    ep_key = f"{sig}_epoch"
    if ep_key not in data:
        print(f"No epoch info for {sig}")
        return
    epochs = data[ep_key]
    counts = defaultdict(int)
    for ep in epochs:
        counts[int(ep)] += 1
    print(f"Per-epoch counts for {sig}:")
    for ep, cnt in sorted(counts.items()):
        print(f"  epoch {ep}: {cnt}")

for sig in signals:
    arr = data[sig]
    summarize_array(arr, sig)
    per_epoch_counts(sig)
    if arr.size:
        plt.figure(figsize=(6, 4))
        sns.histplot(arr, bins=80, kde=False)
        plt.title(f"Histogram of {sig}")
        plt.xlabel(sig)
        plt.ylabel("count")
        plt.tight_layout()
        plt.show()